In [1]:
from langchain.tools import tool
from langchain_groq import ChatGroq
@tool
def get_weather(location: str) -> str:
    """Get the weather for a location."""
    return f"The weather in {location} is sunny."
model = ChatGroq(model="llama-3.1-8b-instant")
model_with_tools = model.bind(tools=[get_weather])


In [ ]:
from langchain.tools import tool
from langchain_groq import ChatGroq

@tool
    def get_weather(location: str) -> str:
        """Get the weather for a location."""
        return f"The weather in {location} is sunny."

model = ChatGroq(model="llama-3.1-8b-instant")

# CHANGE Here: Use bind_tools instead of bind
model_with_tools = model.bind_tools(tools=[get_weather])

# Now this will work perfectly!
response = model_with_tools.invoke("What is the weather in New York?")
print(response.tool_calls)

[{'name': 'get_weather', 'args': {'location': 'New York'}, 'id': 'e6m8kp3tj', 'type': 'tool_call'}]


### Tools 

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

### TOOL EXECUTION LOOP

In [14]:
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage

# Step 1: Set strict rules for the model using a SystemMessage
messages = [
    SystemMessage(content="You are a helpful assistant. You must ONLY use the provided tools. Do NOT attempt to use 'brave_search' or any other unlisted tools. Accept the data provided by get_weather as absolute truth."),
    HumanMessage(content="What is the weather in New York?")
]

ai_message = model_with_tools.invoke(messages)
messages.append(ai_message)

# Step 2: Execute tools and collect results
for tool_call in ai_message.tool_calls:
    
    raw_result = get_weather.invoke(tool_call["args"])
    
    tool_receipt = ToolMessage(
        content=str(raw_result),
        tool_call_id=tool_call["id"],
        name=tool_call["name"]
    )
    
    messages.append(tool_receipt)

# Step 3: Pass the results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.content)

Since the weather in New York is sunny, I will assume that the temperature is also suitable for outdoor activities.


In [15]:
messages

[SystemMessage(content="You are a helpful assistant. You must ONLY use the provided tools. Do NOT attempt to use 'brave_search' or any other unlisted tools. Accept the data provided by get_weather as absolute truth.", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='What is the weather in New York?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '8bgxqgskp', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 262, 'total_tokens': 277, 'completion_time': 0.038772857, 'completion_tokens_details': None, 'prompt_time': 0.115235086, 'prompt_tokens_details': None, 'queue_time': 0.065614232, 'total_time': 0.154007943}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_1151d4f23c', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 